# Quantum Computing Lab - Assignment 5
## Mrudula A Mahindrakar
### EE25S011

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import HamiltonianGate
from qiskit import transpile
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Operator


from qiskit_ibm_runtime.fake_provider import FakeAthensV2
from qiskit_aer.noise import NoiseModel

from scipy.linalg import expm
import warnings
from numpy.linalg import solve, eig

# Problem 1: HHL Algorithm 

To solve the linear system of equation: $A∣x⟩=∣b⟩$

A = $\begin{pmatrix} 1 & 0 & 0 & -1  \\\\ 0 & 1 & 0 & 0 \\\\ 0 & 0 & 1 & 0 \\\\ 1 & 0 & 0 & 1\end{pmatrix}$ 
b = $\begin{pmatrix} 1 \\\\ 0 \\\\ 0 \\\\ 0 \end{pmatrix}$

A is not hermitian, so we define 

C = $\begin{pmatrix} 0 & A \\\\ A^+ & 0 \end{pmatrix}$

So that $\begin{pmatrix} 0 & A \\\\ A^+ & 0 \end{pmatrix}$ . $\begin{pmatrix} 0 \\\\ x \end{pmatrix}$ = $\begin{pmatrix} b \\\\ 0 \end{pmatrix}$

In [2]:
# Define Matrix and Analyze Spectrum
A = np.array([[1, 0, 0, -1], 
                   [0, 1, 0, 0], 
                   [0, 0, 1, 0], 
                   [1, 0, 0, 1]], dtype=complex)

eigvals, eigvecs = eig(A)

print("Eigenvalues:")
print(eigvals)

basis_vectors = [
    np.array([1,0,0,0], dtype=complex),
    np.array([0,1,0,0], dtype=complex),
    np.array([0,0,1,0], dtype=complex),
    np.array([0,0,0,1], dtype=complex)
]

Eigenvalues:
[1.+1.j 1.-1.j 1.+0.j 1.+0.j]


In [3]:
def classical_solution(A, b):
    return solve(A, b)

for i, b in enumerate(basis_vectors):
    x = classical_solution(A, b)
    print(f"Solution for basis vector {i}:")
    print(x)

Solution for basis vector 0:
[ 0.5+0.j  0. +0.j  0. +0.j -0.5+0.j]
Solution for basis vector 1:
[0.+0.j 1.+0.j 0.+0.j 0.+0.j]
Solution for basis vector 2:
[0.+0.j 0.+0.j 1.+0.j 0.+0.j]
Solution for basis vector 3:
[0.5+0.j 0. +0.j 0. +0.j 0.5+0.j]


In [4]:
A_dag = A.conj().T
zeros = np.zeros((4, 4))
C = np.block([[zeros, A], 
              [A_dag, zeros]])

eigvals = np.linalg.eigvals(C)
lambda_max = np.max(np.abs(eigvals))
t = 2 * np.pi * (2**i) / 4 

print("Matrix C is Hermitian:", np.allclose(C, C.conj().T))
print("Hermitian matrix C shape:", C.shape)

Matrix C is Hermitian: True
Hermitian matrix C shape: (8, 8)


In [5]:
def controlled_time_evolution(circuit, C, t, clock, system):
    for i in range(len(clock)):
        U = HamiltonianGate(C, t * (2**i))
        CU = U.control()
        circuit.append(CU, [clock[i]] + system)

In [6]:
def qpe(circuit, C, t, clock, system):
    # Hadamards on clock
    for qubit in clock:
        circuit.h(qubit)
    
    controlled_time_evolution(circuit, C, t, clock, system)

In [7]:
def eigenvalue_rotation(circuit, clock, ancilla, t, C_const):
    n = len(clock)
    
    for k in range(1, 2**n):  # skip 0 to avoid division by zero
        
        # approximate eigenvalue
        lambda_k = 2 * np.pi * k / t
        
        if abs(lambda_k) < 1e-8:
            continue
        
        theta = 2 * np.arcsin(C_const / lambda_k)
        
        # apply multi-controlled rotation
        control_state = format(k, f'0{n}b')
        
        circuit.mcry(
            theta,
            clock,
            ancilla,
            None,
            ctrl_state=control_state
        )

In [8]:
def inverse_qpe(circuit, C, t, clock, system):
    controlled_time_evolution(circuit, C, -t, clock[::-1], system)
    
    for qubit in clock:
        circuit.h(qubit)

In [9]:
from qiskit import QuantumCircuit

def build_hhl(C, b_state):
    n_system = 3  # 8x8 matrix → 3 qubits
    n_clock = 3
    
    qc = QuantumCircuit(n_clock + n_system + 1, 1)
    
    clock = list(range(n_clock))
    system = list(range(n_clock, n_clock + n_system))
    ancilla = n_clock + n_system
    
    # Initialize |b>
    qc.initialize(b_state, system)
    
    # Scale time
    eigvals = np.linalg.eigvals(C)
    lambda_max = np.max(np.abs(eigvals))
    t = 2 * np.pi / lambda_max
    
    # QPE
    qpe(qc, C, t, clock, system)
    
    # Rotation constant
    C_const = 0.5 / lambda_max
    
    eigenvalue_rotation(qc, clock, ancilla, t, C_const)
    
    # Uncompute
    inverse_qpe(qc, C, t, clock, system)
    
    # Measure ancilla
    qc.measure(ancilla, 0)
    
    return qc

In [10]:
b = np.array([1, 0, 0, 0], dtype=complex)

# Normalize
b = b / np.linalg.norm(b)

# Embed into 8D
b_tilde = np.concatenate([b, np.zeros(4)])
qc = build_hhl(C, b_tilde)

sim = AerSimulator()
result = sim.run(qc, shots=8192).result()
counts = result.get_counts()

print(counts)

TypeError: QuantumCircuit.mcry() got an unexpected keyword argument 'ctrl_state'

In [ ]:
statevector = result.get_statevector()
solution = statevector[4:8]

In [ ]:
# def controlled_U(num_phase_qubits, C, qc, qr_c, qr_s):
#     for i in range(num_phase_qubits):
#         # Time evolution factor t = 2*pi / max_eigenvalue
#         t = 2 * np.pi * (2**i) / 4 
#         u_gate = HamiltonianGate(C, t).control()
#         qc.append(u_gate, [qr_c[i]] + qr_s[:])

In [ ]:
def controlled_time_evolution(circuit, C, t, clock, system):
    for i in range(len(clock)):
        U = HamiltonianGate(C, t * (2**i))
        CU = U.control()
        circuit.append(CU, [clock[i]] + system)

In [ ]:
def qpe(circuit, C, t, clock, system):
    # Hadamards on clock
    for qubit in clock:
        circuit.h(qubit)
    
    controlled_time_evolution(circuit, C, t, clock, system)

In [ ]:
def qpe(circ, clock, system, A):
    circ.h(clock)
    
    # Controlled Hamiltonian Evolution e^{iAt}
    for i in range(len(clock)):
        # t = 2 * pi * 2^i / total_time_scale
        t = 2 * np.pi * (2**i) / 4 
        u_gate = HamiltonianGate(A, t).control()
        circ.append(u_gate, [clock[i]] + list(system))
        
    # Inverse QFT to move phase to register
    circ.append(QFT(len(clock)).inverse(), clock)

In [ ]:
def inv_qpe(circ, clock, system, C):
    # QFT
    circ.append(QFT(len(clock)), clock)
    
    # Reverse Controlled-U
    for i in reversed(range(len(clock))):
        #t = 2 * np.pi * (2**i) / 4
        t = 2 * np.pi / lambda_max
        u_gate = HamiltonianGate(C, -t).control()
        circ.append(u_gate, [clock[i]] + list(system))

    circ.h(clock)

In [ ]:
def conditional_rotation(circ, clock, ancilla):
    circ.cry(np.pi, clock[0], ancilla)
    circ.cry(np.pi/4, clock[1], ancilla)

In [ ]:
def build_hhl_circuit(C, b):
    n_system = 3          # 8-dimensional space
    n_clock = 3           # precision qubits
    
    # 3 qubits for system (8x8 matrix), 2 for clock, 1 
    
    qr_a = QuantumRegister(1, 'ancilla')
    qr_c = QuantumRegister(n_clock, 'clock')
    qr_s = QuantumRegister(n_system, 'system')
    cr = ClassicalRegister(4, 'meas')
    
    qc = QuantumCircuit(qr_a, qr_c, qr_s, cr)

    b_padded = np.concatenate([b, np.zeros(4)])
    b_padded = b_padded / np.linalg.norm(b_padded)
    
    qc.initialize(b_padded, qr_s)
    
    qpe(qc, qr_c, qr_s, C)
    
    conditional_rotation(qc, qr_c, qr_a)

    inv_qpe(qc, qr_c, qr_s, C)
    
    qc.measure(qr_a, cr[0])
    qc.measure(qr_s, cr[1:4])
    
    
    return qc

In [ ]:
warnings.filterwarnings("ignore", category=DeprecationWarning)

qc = build_hhl_circuit(C, b)

qc.draw("mpl")

In [ ]:
def run_experiment(sim_type, simulator):
    print(f"\n--- Running on {sim_type} Simulator ---")
    for i, b in enumerate(basis_vectors):
        # Build the scratch circuit
        qc = build_hhl_circuit(C, b)
        
        # Transpile and Run
        t_qc = transpile(qc, simulator)
        result = simulator.run(t_qc, shots=8192).result()
        counts = result.get_counts()
        
        print(f"Basis Vector b_{i} result (Counts): {counts}")

    return result

In [ ]:
ideal_sim = AerSimulator()

warnings.filterwarnings("ignore", category=DeprecationWarning)

Ideal_result = run_experiment("Ideal", ideal_sim)

counts_circuit_ideal = Ideal_result.get_counts()

display(plot_histogram(counts_circuit_ideal))


In [ ]:
fake_backend = FakeAthensV2()
noise_model = NoiseModel.from_backend(fake_backend)
noisy_sim = AerSimulator(noise_model=noise_model)

warnings.filterwarnings("ignore", category=DeprecationWarning)

Noisy_result = run_experiment("Noisy", noisy_sim)
counts_circuit_noisy = Noisy_result.get_counts()

display(plot_histogram(counts_circuit_noisy))

In [ ]:
from scipy.linalg import expm
from qiskit.quantum_info import Operator

def controlled_U(circuit, clock, system, A, t=1):
    """
    Applies controlled e^{iAt}
    clock  : phase register (list of qubits)
    system : system register (target qubit)
    """

    # Compute unitary e^{iAt}
    U = expm(1j * A * t)

    # Convert to Qiskit Operator
    U_op = Operator(U)

    # Convert to instruction
    U_gate = U_op.to_instruction()

    # Create multi-controlled version
    cU = U_gate.control(num_ctrl_qubits=len(clock))

    # Append to circuit
    circuit.append(cU, list(clock) + [system[0]]) # Control on clock, target is system[0]

In [ ]:
# Create the Quantum and Classical registers needed
clock = QuantumRegister(2, name='clock') # Register to hold the eigenvalues
b_vec = QuantumRegister(1, name='b') # Register to hold the input vector b
ancilla = QuantumRegister(1, name='ancilla') # Ancilla qubit for the controlled rotation
measurement = ClassicalRegister(2, name='c') # Classical register to hold the measurement results of the clock register

# Create an empty circuit with the specified registers
circuit = QuantumCircuit(ancilla, clock, b_vec, measurement)

circuit.draw(output='mpl')

In [ ]:
def qft(circ, q, n):
    """
    Apply Quantum Fourier Transform on qubits q (little-endian order).
    q[0] = least significant qubit.
    """

    # Apply QFT rotations
    for j in range(n):
        circ.h(q[j])
        for k in range(j+1, n):
            circ.cp(np.pi / (2**(k-j)), q[k], q[j])

    # Reverse qubit order
    for j in range(n // 2):
        circ.swap(q[j], q[n - j - 1])

def qft_dagger(circ, q, n):
    """
    Apply inverse Quantum Fourier Transform (QFT†)
    q[0] = least significant qubit (Qiskit little-endian)
    """

    # Step 1: Reverse qubit order
    for j in range(n // 2):
        circ.swap(q[j], q[n - j - 1])

    # Step 2: Apply inverse rotations
    for j in reversed(range(n)):
        for k in reversed(range(j+1, n)):
            circ.cp(-np.pi / (2**(k-j)), q[k], q[j])
        circ.h(q[j])

In [ ]:
# draw qft and qft_dagger circuits
qft_circuit = QuantumCircuit(clock)
qft(qft_circuit, clock, len(clock))
qft_circuit.draw(output='mpl')

In [ ]:
qft_dagger_circuit = QuantumCircuit(clock)
qft_dagger(qft_dagger_circuit, clock, len(clock))
qft_dagger_circuit.draw(output='mpl')

In [ ]:
def qpe(circuit, clock, b_vec):
    '''
    Performs quantum phase estimation to find the eigenvalues of the matrix A.
    '''
    # Perform a Hadamard Transform
    circuit.h(clock)


    # e^{i*A*t} # Apply the controlled unitary operation for time t=1
    circuit.cu(np.pi/2, -np.pi/2, np.pi/2, 3*np.pi/4, clock[0], b_vec, label='U'); 
    # controlled_U(circuit, clock, b_vec, A, t=1)
    
    # e^{i*A*t*2} # Apply the controlled unitary operation for time t=2
    circuit.cu(np.pi, np.pi, 0, 0, clock[1], b_vec, label='U2');
    # controlled_U(circuit, clock, b_vec, A, t=2)
    
    # Perform an inverse QFT on the register holding the eigenvalues
    qft_dagger(circuit, clock, 2)
    
def inv_qpe(circuit, clock, b_vec):
    
    # Perform a QFT on the register holding the eigenvalues
    qft(circuit, clock, 2)

    # e^{i*A*t*2} 
    circuit.cu(np.pi, np.pi, 0, 0, clock[1], b_vec, label='U2');
    # controlled_U(circuit, clock, b_vec, A, t=2)

    # e^{i*A*t}
    circuit.cu(np.pi/2, -np.pi/2, np.pi/2, 3*np.pi/4, clock[0], b_vec, label='U');
    # controlled_U(circuit, clock, b_vec, A, t=1)

    # Perform a Hadamard Transform
    circuit.h(clock)

In [ ]:
# draw the circuit for the phase estimation step
qpe_circuit = QuantumCircuit(ancilla, clock, b_vec)
qpe(qpe_circuit, clock, b_vec)
qpe_circuit.draw(output='mpl')

In [ ]:
inverse_qpe_circuit = QuantumCircuit(ancilla, clock, b_vec)
inv_qpe(inverse_qpe_circuit, clock, b_vec)
inverse_qpe_circuit.draw(output='mpl')

In [ ]:
# Function to build the HHL circuit from the major components 
def hhl(circ, ancilla, clock, b_vec, measurement):
    
    qpe(circ, clock, b_vec)
    
    #conditional Rotation
    circuit.cry(np.pi, clock[0], ancilla)
    circuit.cry(np.pi/3, clock[1], ancilla)
    
    circuit.measure(ancilla, measurement[0])
    inv_qpe(circ, clock, b_vec)

In [ ]:
# State preparation. 
intial_state = b # The state |b> is prepared in the b_vec register
circuit.initialize(intial_state, 4)

hhl(circuit, ancilla, clock, b_vec, measurement)

circuit.measure(b_vec, measurement[1])

In [ ]:
circuit.draw('mpl',scale=1)